# Summarizing Data

In [17]:
import os
from dotenv import load_dotenv

import pandas as pd
import sqlalchemy

In [18]:
load_dotenv()

db_host = os.environ.get("db_host")
db_user = os.environ.get("db_user")
db_password = os.environ.get("db_password")

In [19]:
engine = sqlalchemy.create_engine(f"mysql+pymysql://{db_user}:{db_password}@{db_host}:3306/sql_invoicing")

In [20]:
pd.read_sql("show tables", con= engine)

,Tables_in_sql_invoicing
0,clients
1,invoices
2,payment_methods
3,payments


## Aggregate Functions

In [21]:
query = """
select
	max(invoice_total) as highest,
    max(invoice_total) as lowest,
    avg(invoice_total) as average,
    sum(invoice_total) as total,
	count(invoice_total) as number_of_invoices,
    count(payment_date) as number_of_payments
    
from invoices
"""

pd.read_sql(query, con=engine)

,highest,lowest,average,total,number_of_invoices,number_of_payments
0,189.12,189.12,152.388235,2590.6,17,7


EXERCISE:

In [22]:
query = """
select 
	"first half of 2019" as date_range,
    sum(invoice_total) as total_sales,
    sum(payment_total) as total_payments,
     sum(invoice_total - payment_total) as what_we_expect
from invoices
where invoice_date between '2019-01-01' and '2019-06-30'
union
select 
	"second half of 2019" as date_range,
    sum(invoice_total) as total_sales,
    sum(payment_total) as total_payments,
     sum(invoice_total - payment_total) as what_we_expect
from invoices
where invoice_date between '2019-07-01' and '2019-12-31'
union
select 
	"total" as date_range,
    sum(invoice_total) as total_sales,
    sum(payment_total) as total_payments,
     sum(invoice_total - payment_total) as what_we_expect
from invoices
"""

pd.read_sql(query, con=engine)

,date_range,total_sales,total_payments,what_we_expect
0,first half of 2019,1539.07,212.97,1326.10
1,second half of 2019,1051.53,148.41,903.12
2,total,2590.60,361.38,2229.22


## GROUP BY

In [23]:
query = """
select 
	client_id,
    sum(invoice_total)
from invoices
group by client_id
"""

pd.read_sql(query, con=engine)

,client_id,sum(invoice_total)
0,1,802.89
1,2,101.79
2,3,705.90
3,5,980.02


we can group data by the combination of different parameter. see the example below:

In [24]:
query = """
select 
	state,
    city,
    sum(invoice_total)
from invoices i
join clients c using (client_id)
group by state, city

-- we get one record for each state and city combination
"""

pd.read_sql(query, con=engine)

,state,city,sum(invoice_total)
0,WV,Huntington,101.79
1,OR,Portland,980.02
2,CA,San Francisco,705.90
3,NY,Syracuse,802.89


EXERCISE:

In [25]:
query = """
select 
	p.date,
    pm.name as payment_method,
    sum(p.amount) as total_payments

from payments p
join payment_methods pm
	on p.payment_method = pm.payment_method_id

group by p.date, p.payment_method
order by p.date	
"""

pd.read_sql(query, con=engine)

,date,payment_method,total_payments
0,2019-01-03,Credit Card,74.55
1,2019-01-08,Credit Card,32.77
2,2019-01-08,Cash,10.00
3,2019-01-11,Credit Card,0.03
4,2019-01-15,Credit Card,148.41
5,2019-01-26,Credit Card,87.44
6,2019-02-12,Credit Card,8.18


## HAVING
with the `WHERE` clause, we can filter data before our rows are groups and with the `HAVING` clause, we can filter data after our rows are grouped.

notice that the columns that we use in `HAVING` clause, have to be in the `SELECT` clause; but in the `WHERE` clause, we can reference any columns whether we are selecting them or not. 

In [26]:
query = """
select 
	client_id,
    sum(invoice_total) as total_sales
from invoices
group by client_id
having total_sales > 500
"""

pd.read_sql(query, con=engine)

,client_id,total_sales
0,1,802.89
1,3,705.90
2,5,980.02


EXERCISE:

In [28]:
query = """
select 
	c.customer_id,
    c.first_name, 
    c.last_name,
    c.state,
	sum(oi.quantity * oi.unit_price) as total_spent

from sql_store.customers c

join sql_store.orders o
	using (customer_id)

join sql_store.order_items oi
	using(order_id)

where c.state = 'VA'

group by 
	c.customer_id,
    c.first_name, 
    c.last_name

having total_spent > 100	
"""

pd.read_sql(query, con=engine)

,customer_id,first_name,last_name,state,total_spent
0,2,Ines,Brushfield,VA,157.92


## ROLLUP

In [30]:
query = """
select 
	client_id,
    sum(invoice_total)
from invoices
group by client_id with rollup
"""

pd.read_sql(query, con=engine)

,client_id,sum(invoice_total)
0,1.0,802.89
1,2.0,101.79
2,3.0,705.90
3,5.0,980.02
4,NaN,2590.60


In [29]:
query = """
select 
	state,
    city,
    sum(invoice_total)
from invoices i
join clients c using (client_id)
group by state, city with rollup
-- we get one record for each state and city combination
"""

pd.read_sql(query, con=engine)

,state,city,sum(invoice_total)
0,CA,San Francisco,705.90
1,CA,None,705.90
2,NY,Syracuse,802.89
3,NY,None,802.89
4,OR,Portland,980.02
5,OR,None,980.02
6,WV,Huntington,101.79
7,WV,None,101.79
8,None,None,2590.60


EXERCISE:

In [31]:
query = """
select 
	pm.name as payment_method,
    sum(p.amount) as total
from payments p
join payment_methods pm
	on p.payment_method = pm.payment_method_id
group by pm.name with rollup
"""

pd.read_sql(query, con=engine)

,payment_method,total
0,Cash,10.00
1,Credit Card,351.38
2,None,361.38
